In [0]:
# Questo notebook aggiorna giornalmente la Bronze ENTSO-E.
#
# Scarica da ENTSO-E solo il giorno appena concluso
# e aggiorna la tabella Bronze:
# progetto_meteo.bronze.dati_entsoe
#
# Il notebook viene usato nel JOB_AGGIORNAMENTO_GIORNALIERO.
# Dopo questo notebook viene eseguito Patcher_Energy_Block,
# che aggrega il giorno appena scaricato nella tabella Silver energy_block.
#
# Logica importante:
# - scarico solo il giorno appena concluso;
# Gli orari ENTSO-E arrivano in UTC.
# Nel progetto vengono convertiti in Europe/Rome per restare coerenti con Open-Meteo.
#
# Il token ENTSO-E non è scritto nel codice.
# Viene letto da Databricks nel Secret

import requests
import xml.etree.ElementTree as ET

from time import sleep
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType
)


# Imposto Spark in UTC.
# Questo aiuta a gestire correttamente la conversione successiva verso Europe/Rome.
spark.conf.set("spark.sql.session.timeZone", "UTC")


# Imposto catalogo, schemi e tabelle del progetto.
catalogo = "progetto_meteo"
schema_metadati = "metadati"
schema_bronze = "bronze"

tabella_zone_entsoe = f"{catalogo}.{schema_metadati}.zone_entsoe"
tabella_entsoe = f"{catalogo}.{schema_bronze}.dati_entsoe"


# Seleziono il catalogo del progetto.
spark.sql(f"USE CATALOG {catalogo}")


# Leggo il token ENTSO-E da Databricks Secrets.
# In questo modo il token non viene scritto nel notebook e il progetto resta pronto per GitHub.
token_entsoe = dbutils.secrets.get(
    scope="progetto-meteo",
    key="entsoe-token"
)


# Calcolo il giorno appena concluso.
# Se il notebook parte oggi, scarica ieri dalle 00:00 alle 00:00 di oggi.
fuso_italia = ZoneInfo("Europe/Rome")

oggi = datetime.now(fuso_italia).date()
giorno_da_scaricare = oggi - timedelta(days=1)

inizio = datetime(
    giorno_da_scaricare.year,
    giorno_da_scaricare.month,
    giorno_da_scaricare.day,
    0,
    0,
    tzinfo=fuso_italia
)

fine = inizio + timedelta(days=1)


# Creo due stringhe di filtro in orario italiano.
# Servono dopo la conversione da UTC a Europe/Rome, per tenere solo il giorno corretto.
inizio_filtro = inizio.replace(tzinfo=None).strftime("%Y-%m-%d %H:%M:%S")
fine_filtro = fine.replace(tzinfo=None).strftime("%Y-%m-%d %H:%M:%S")

print(f"Giorno ENTSO-E da aggiornare: {giorno_da_scaricare}")
print(f"Periodo ENTSO-E: da {inizio_filtro} a {fine_filtro} fine esclusiva")


# Tengo solo i tipi di produzione utili per il progetto.
# Nel layer Silver verranno aggregati in:
# - Solare
# - Idrico
# - Eolico
tipi_utili = {
    "B16": "Solar",
    "B11": "Hydro Run-of-river and pondage",
    "B12": "Hydro Water Reservoir",
    "B18": "Wind Offshore",
    "B19": "Wind Onshore"
}


# Definisco lo schema temporaneo dei dati estratti dall'XML.
# In questa fase gli orari sono ancora stringhe testuali UTC.
schema_grezzo = StructType([
    StructField("Zona", StringType(), True),
    StructField("Area", StringType(), True),
    StructField("Codice_Entsoe", StringType(), True),
    StructField("Tipo_Produzione", StringType(), True),
    StructField("Start_UTC_Testo", StringType(), True),
    StructField("Fine_UTC_Testo", StringType(), True),
    StructField("Produzione", DoubleType(), True),
    StructField("Timestamp_Caricamento_Testo", StringType(), True)
])


# Converto una data locale Europe/Rome nel formato UTC richiesto da ENTSO-E.
def formato_entsoe(data_italia):
    return data_italia.astimezone(timezone.utc).strftime("%Y%m%d%H%M")


# Leggo un valore XML ignorando il namespace.
# ENTSO-E usa namespace XML, quindi controllo solo la parte finale del tag.
def leggi_testo(elemento, nome):

    for figlio in elemento.iter():
        if figlio.tag.endswith(nome):
            return figlio.text

    return None


# Converto la resolution ENTSO-E in minuti.
# Le risoluzioni gestite dal progetto sono 15, 30 e 60 minuti.
def minuti_resolution(resolution):

    if resolution == "PT15M":
        return 15

    if resolution == "PT30M":
        return 30

    if resolution in ["PT60M", "PT1H"]:
        return 60

    raise Exception(f"Resolution non gestita: {resolution}")


# Chiamo l'API ENTSO-E con piccoli retry.
# Se una chiamata fallisce temporaneamente, riprovo fino a 3 volte.
def chiama_entsoe(parametri):

    endpoint = "https://web-api.tp.entsoe.eu/api"

    for tentativo in range(1, 4):

        try:
            risposta = requests.get(endpoint, params=parametri, timeout=120)
            risposta.raise_for_status()
            return risposta.text

        except Exception as errore:

            if tentativo == 3:
                raise errore

            print(f"Tentativo {tentativo} fallito. Riprovo tra 5 secondi.")
            sleep(5)


# Estraggo dall'XML solo le righe utili al progetto.
# Ogni punto ENTSO-E viene trasformato in una riga con:
# - zona;
# - area;
# - tipo produzione;
# - orario di inizio e fine in UTC;
# - produzione.
def estrai_righe(xml_testo, zona, area, codice_entsoe):

    righe = []
    radice = ET.fromstring(xml_testo)
    timestamp_caricamento = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")

    for serie in radice.iter():

        if not serie.tag.endswith("TimeSeries"):
            continue

        codice_tipo = leggi_testo(serie, "psrType")

        if codice_tipo not in tipi_utili:
            continue

        for periodo in serie.iter():

            if not periodo.tag.endswith("Period"):
                continue

            start_testo = leggi_testo(periodo, "start")
            resolution = leggi_testo(periodo, "resolution")

            if start_testo is None or resolution is None:
                continue

            start_periodo = datetime.fromisoformat(start_testo.replace("Z", "+00:00"))
            minuti = minuti_resolution(resolution)

            for punto in periodo.iter():

                if not punto.tag.endswith("Point"):
                    continue

                posizione_testo = leggi_testo(punto, "position")
                produzione_testo = leggi_testo(punto, "quantity")

                if posizione_testo is None:
                    continue

                try:
                    posizione = int(posizione_testo)
                except:
                    continue

                try:
                    produzione = float(produzione_testo) if produzione_testo is not None else None
                except:
                    produzione = None

                start_utc = start_periodo + timedelta(minutes=(posizione - 1) * minuti)
                fine_utc = start_utc + timedelta(minutes=minuti)

                righe.append({
                    "Zona": zona,
                    "Area": area,
                    "Codice_Entsoe": codice_entsoe,
                    "Tipo_Produzione": tipi_utili[codice_tipo],
                    "Start_UTC_Testo": start_utc.strftime("%Y-%m-%d %H:%M:%S"),
                    "Fine_UTC_Testo": fine_utc.strftime("%Y-%m-%d %H:%M:%S"),
                    "Produzione": produzione,
                    "Timestamp_Caricamento_Testo": timestamp_caricamento
                })

    return righe


# Leggo dalla tabella metadati le zone ENTSO-E da scaricare.
zone_entsoe = [
    riga.asDict()
    for riga in (
        spark.table(tabella_zone_entsoe)
        .select("Zona", "Area", "Codice_Entsoe")
        .orderBy("Zona")
        .collect()
    )
]

print(f"Zone ENTSO-E da scaricare: {len(zone_entsoe)}")


# Scarico il giorno appena concluso per tutte le zone ENTSO-E.
righe_totali = []

for zona in zone_entsoe:

    print(f"Scarico {zona['Zona']} per il giorno {giorno_da_scaricare}")

    parametri = {
        "securityToken": token_entsoe,
        "documentType": "A75",
        "processType": "A16",
        "in_Domain": zona["Codice_Entsoe"],
        "periodStart": formato_entsoe(inizio),
        "periodEnd": formato_entsoe(fine)
    }

    xml_testo = chiama_entsoe(parametri)

    righe = estrai_righe(
        xml_testo=xml_testo,
        zona=zona["Zona"],
        area=zona["Area"],
        codice_entsoe=zona["Codice_Entsoe"]
    )

    if len(righe) == 0:
        print(f"Nessuna riga utile trovata per {zona['Zona']}.")
    else:
        righe_totali.extend(righe)
        print(f"Righe trovate per {zona['Zona']}: {len(righe)}")

    sleep(0.2)


# Controllo che siano stati scaricati dati prima di toccare la Bronze.
# Se non è arrivato nulla dall'API, non modifico bronze.dati_entsoe.
if len(righe_totali) == 0:
    raise Exception("Nessun dato ENTSO-E scaricato. Non modifico bronze.dati_entsoe.")


# Creo il DataFrame grezzo usando lo schema esplicito.
df_grezzo = spark.createDataFrame(righe_totali, schema=schema_grezzo)


# Converto orari e colonne nel formato finale della Bronze.
# Mantengo gli orari UTC e aggiungo gli orari convertiti in Europe/Rome.
df_entsoe_nuovo = (
    df_grezzo
    .withColumn("Start_UTC", F.to_timestamp("Start_UTC_Testo"))
    .withColumn("Fine_UTC", F.to_timestamp("Fine_UTC_Testo"))
    .withColumn("Start", F.from_utc_timestamp("Start_UTC", "Europe/Rome"))
    .withColumn("Fine", F.from_utc_timestamp("Fine_UTC", "Europe/Rome"))
    .withColumn("Data", F.to_date("Start"))
    .withColumn("Ora", F.hour("Start").cast("bigint"))
    .withColumn("Timestamp_Caricamento", F.to_timestamp("Timestamp_Caricamento_Testo"))
    .filter(F.col("Start") >= F.lit(inizio_filtro))
    .filter(F.col("Start") < F.lit(fine_filtro))
    .select(
        "Zona",
        "Area",
        "Codice_Entsoe",
        "Tipo_Produzione",
        "Start_UTC",
        "Fine_UTC",
        "Start",
        "Fine",
        "Data",
        "Ora",
        "Produzione",
        "Timestamp_Caricamento"
    )
)


# Controllo le righe effettive dopo i filtri.
# Se i dati scaricati non rientrano nel giorno richiesto, non aggiorno la Bronze.
righe_nuove = df_entsoe_nuovo.count()

if righe_nuove == 0:
    raise Exception("Dati scaricati, ma nessuna riga valida dopo i filtri. Non modifico bronze.dati_entsoe.")


# Sovrascrivo solo il giorno appena concluso nella Bronze.
# Non faccio DELETE manuale e non sovrascrivo tutta la tabella.
predicato_bronze = f"Data = '{giorno_da_scaricare}'"

df_entsoe_nuovo.write.mode("overwrite").format("delta").option("replaceWhere", predicato_bronze).saveAsTable(
    tabella_entsoe
)


# Controllo finale.
# Mostro un campione ordinato dei dati appena aggiornati.
display(
    df_entsoe_nuovo
    .orderBy("Zona", "Tipo_Produzione", "Start")
    .limit(100)
)


# Stampo un riepilogo finale dello streaming giornaliero ENTSO-E.
print("Streaming ENTSO-E giornaliero completato.")
print(f"Giorno aggiornato in Bronze: {giorno_da_scaricare}")
print(f"Tabella aggiornata: {tabella_entsoe}")
print(f"Righe scritte in Bronze: {righe_nuove}")